# 02 — Feature Engineering
**Credit Risk Scoring Pipeline | Proje 1**

This notebook applies all feature engineering steps on the cleaned data.
It uses the modules from `src/credit_risk/` — no raw logic is written here.

Steps:
1. Load data and apply preprocessor
2. Create domain features (engineer.py)
3. WoE/IV encoding (woe_encoder.py)
4. Feature correlation with TARGET
5. Save processed data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Add project root to path so we can import from src/
sys.path.insert(0, os.path.abspath('..'))

from src.credit_risk.data.preprocessor import CreditRiskPreprocessor
from src.credit_risk.features.engineer import CreditRiskFeatureEngineer
from src.credit_risk.features.woe_encoder import WoEEncoder

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('All imports OK')

## Step 1 — Load Data & Apply Preprocessor

In [ ]:
df = pd.read_csv('../data/raw/application_train.csv')
print(f'Raw data shape: {df.shape}')

# Separate features and target before fitting
# IMPORTANT: preprocessor is fit on X only, never on y
X = df.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df['TARGET']

# MNAR columns identified in EDA:
# OWN_CAR_AGE  → missing means customer has no car
# DAYS_EMPLOYED == 365243 → means not employed (handled in engineer.py via clip)
preprocessor = CreditRiskPreprocessor(
    missing_threshold=0.5,
    mnar_cols=['OWN_CAR_AGE']
)

X_clean = preprocessor.fit_transform(X)
print(f'After preprocessing: {X_clean.shape}')
print(f'Columns dropped: {len(preprocessor.drop_cols_)}')
print(f'Remaining nulls: {X_clean.isnull().sum().sum()}')

## Step 2 — Domain Feature Engineering

In [ ]:
engineer = CreditRiskFeatureEngineer()
X_feat = engineer.fit_transform(X_clean)

# New features created
new_features = [
    'DTI_RATIO', 'CREDIT_INCOME_RATIO', 'CREDIT_GOODS_RATIO',
    'ANNUITY_CREDIT_RATIO', 'AGE_YEARS', 'EMPLOYMENT_YEARS',
    'EMPLOYMENT_AGE_RATIO', 'INCOME_PER_FAMILY', 'AGE_GROUP'
]
print(f'Shape after feature engineering: {X_feat.shape}')
print(f'New features added: {len(new_features)}')
X_feat[new_features].describe().round(3)

In [ ]:
# Check correlation of new features with TARGET
X_feat_with_target = X_feat.copy()
X_feat_with_target['TARGET'] = y.values

numeric_new = [f for f in new_features if f != 'AGE_GROUP']
corr_new = X_feat_with_target[numeric_new + ['TARGET']].corr()['TARGET'].drop('TARGET')
corr_new = corr_new.abs().sort_values(ascending=False)

print('New feature correlation with TARGET:')
print(corr_new.round(4))

# Visualise
fig, ax = plt.subplots(figsize=(10, 5))
corr_new.plot(kind='barh', ax=ax, color='#7c3aed', alpha=0.8)
ax.set_xlabel('|Correlation with TARGET|')
ax.set_title('New Feature Correlation with Default')
plt.tight_layout()
plt.savefig('../outputs/figures/new_feature_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 3 — WoE / IV Encoding

In [ ]:
# Select categorical columns for WoE encoding
cat_cols = X_feat.select_dtypes(include=['object', 'category']).columns.tolist()
# Remove AGE_GROUP — it is already ordinal, keep as-is
cat_cols_for_woe = [c for c in cat_cols if c != 'AGE_GROUP']
print(f'Columns to WoE encode: {len(cat_cols_for_woe)}')

woe = WoEEncoder()
woe.fit(X_feat, y, columns=cat_cols_for_woe)

# IV summary — shows which features are worth keeping
iv_summary = woe.get_iv_summary()
print('\nIV Summary:')
print(iv_summary.to_string(index=False))

In [ ]:
# Visualise IV scores
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#16a34a' if iv > 0.3 else '#2563eb' if iv > 0.1 else '#f59e0b' if iv > 0.02 else '#dc2626'
          for iv in iv_summary['IV']]
ax.barh(iv_summary['Feature'], iv_summary['IV'], color=colors, alpha=0.85)
ax.axvline(0.02, color='gray', ls='--', alpha=0.5, label='Weak (0.02)')
ax.axvline(0.1,  color='blue', ls='--', alpha=0.5, label='Medium (0.1)')
ax.axvline(0.3,  color='green', ls='--', alpha=0.5, label='Strong (0.3)')
ax.set_xlabel('Information Value (IV)')
ax.set_title('Feature Predictive Power (IV Score)')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/iv_scores.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Apply WoE transformation
X_woe = woe.transform(X_feat)
print(f'Shape after WoE encoding: {X_woe.shape}')

# Drop original categorical columns — WoE versions replace them
X_final = X_woe.drop(columns=cat_cols_for_woe, errors='ignore')
X_final['TARGET'] = y.values
print(f'Final feature set shape: {X_final.shape}')

## Step 4 — Log Transforms for Skewed Features

In [ ]:
# Apply log1p to highly skewed columns identified in EDA
# log1p(x) = log(1+x) — safe for zero values
skewed_cols = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE']
skewed_cols = [c for c in skewed_cols if c in X_final.columns]

for col in skewed_cols:
    X_final[f'{col}_LOG'] = np.log1p(X_final[col])
    print(f'{col} skew before: {X_final[col].skew():.2f}  →  after log: {X_final[f"{col}_LOG"].skew():.2f}')

## Step 5 — Save Processed Data

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

X_final.to_csv('../data/processed/features_engineered.csv', index=False)
iv_summary.to_csv('../outputs/reports/iv_summary.csv', index=False)

print(f'Saved features_engineered.csv — shape: {X_final.shape}')
print(f'Saved iv_summary.csv — {len(iv_summary)} features')
print('\nFeature engineering complete. Ready for 03_modeling.ipynb')